## Local pipeline extension

## Goal : compare the existing linear regression baseline with a ridge regression model

## both models use 
## 1 the same training data
## 3 the same features 
## 2 the same train/validation 
## 4 RMSE as the evaluation metric



In [29]:
import mlflow
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from src.batch.flow import score_batch_flow

from src.common.config import get_settings
from src.common.features import (
    TARGET_COLUMN,
    load_dataframe,
    prepare_dataframe,
    to_model_records,
)
from src.common.model_registry import configure_tracking, ensure_experiment

from pathlib import Path

output_path = Path(
    "data/predictions/notebook_batch_predictions.parquet"
)
from fastapi.testclient import TestClient
from src.serve.api import app

In [2]:
settings = get_settings()
settings

ProjectSettings(mlflow_tracking_uri='http://127.0.0.1:5001', prefect_api_url='http://127.0.0.1:4200/api', experiment_name='green-taxi-duration', registered_model_name='green-taxi-duration', model_uri=None, model_alias=None, model_stage=None, model_version=None, run_id=None, model_artifact_path='model', train_data_uri='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet', batch_input_uri='https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-02.parquet', batch_output_dir=PosixPath('/Users/youssef/projects/bootcamp/mle-batch-and-stream-predictions/data/predictions'), artifacts_dir=PosixPath('/Users/youssef/projects/bootcamp/mle-batch-and-stream-predictions/storage/mlartifacts'))

load and inspect training data

In [3]:
df_raw = load_dataframe(settings.train_data_uri)
df_raw.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-01-01 00:03:01,2025-01-01 00:17:12,N,1.0,75,235,1.0,5.93,24.70,1.0,0.5,6.80,0.00,NaN,1.0,34.00,1.0,1.0,0.00,0.0
1,2,2025-01-01 00:19:59,2025-01-01 00:25:52,N,1.0,166,75,1.0,1.32,8.60,1.0,0.5,0.00,0.00,NaN,1.0,11.10,2.0,1.0,0.00,0.0
2,2,2025-01-01 00:05:29,2025-01-01 00:07:21,N,5.0,171,73,1.0,0.41,25.55,0.0,0.0,0.00,0.00,NaN,1.0,26.55,2.0,2.0,0.00,0.0
3,2,2025-01-01 00:52:24,2025-01-01 01:07:52,N,1.0,74,223,1.0,4.12,21.20,1.0,0.5,6.13,6.94,NaN,1.0,36.77,1.0,1.0,0.00,0.0
4,2,2025-01-01 00:25:05,2025-01-01 01:01:10,N,1.0,66,158,1.0,4.71,33.80,1.0,0.5,7.81,0.00,NaN,1.0,46.86,1.0,1.0,2.75,0.0


In [4]:
prepared = prepare_dataframe(df_raw, include_target=True)
print(f"Rows before/after filtering: {len(df_raw):,} / {len(prepared):,}")
prepared[["trip_route", "trip_distance", TARGET_COLUMN]].head()

Rows before/after filtering: 48,326 / 44,525


,trip_route,trip_distance,trip_duration_minutes
0,75_235,5.93,14.183333
1,166_75,1.32,5.883333
2,171_73,0.41,1.866667
3,74_223,4.12,15.466667
4,66_158,4.71,36.083333


## Train/Validation Split

In [10]:
records = to_model_records(prepared)

target = prepared[TARGET_COLUMN]

X_train, X_valid , y_train, y_valid = train_test_split(

    records,
    target,
    test_size = 0.2,
    random_state = 42,

)
len(X_train), len(X_valid)


(35620, 8905)

##  Linear Regression Baseline

In [12]:
linear_pipeline = make_pipeline(
    DictVectorizer(),
    LinearRegression(),
)

linear_pipeline.fit(X_train,y_train)
linear_predictions = linear_pipeline.predict(X_valid)

linear_rmse = root_mean_squared_error(
    y_valid,
    linear_predictions,
)

print(f"Linear Regression RMSE: {linear_rmse:.2f}")

Linear Regression RMSE: 4.80


## Ridge pipeline 

In [14]:
ridge_pipeline = make_pipeline(

    DictVectorizer(),
    Ridge(),
)

ridge_pipeline.fit(X_train, y_train)

ridge_predictions = ridge_pipeline.predict(X_valid)
ridge_rmse = root_mean_squared_error(

    y_valid,
    ridge_predictions,
)

print(f"Fidge Regression RMSE : {ridge_rmse:.2f}")

Fidge Regression RMSE : 4.67


### comparaison

In [16]:
comparaison = pd.DataFrame(

    {
        "model" : [
            "Linear Regression",
            "Ridge Regression",

        ],
        "rmse" : [
            linear_rmse,
            ridge_rmse,
        ]


    }


)
comparaison

,model,rmse
0,Linear Regression,4.800434
1,Ridge Regression,4.673217


## MLFlow

In [17]:
configure_tracking(settings.mlflow_tracking_uri)

experiment = ensure_experiment(settings.experiment_name)

experiment

'1'

## Log the first MLFlow for linear regression

In [19]:
with mlflow.start_run(
    experiment_id=experiment,
    run_name="linear-regression",
):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", linear_rmse)

    mlflow.sklearn.log_model(
        sk_model=linear_pipeline,
        name=settings.model_artifact_path,
        registered_model_name=settings.registered_model_name,
        serialization_format="skops",
    )

Registered model 'green-taxi-duration' already exists. Creating a new version of this model...
2026/09/24 17:37:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: green-taxi-duration, version 3
Created version '3' of model 'green-taxi-duration'.


🏃 View run linear-regression at: http://127.0.0.1:5001/#/experiments/1/runs/6f3a4428b6d148e1884805119432787d
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


## for Ridge Regression

In [20]:
with mlflow.start_run(
    experiment_id=experiment,
    run_name="ridge-regression",
):
    mlflow.log_param("model_type", "Ridge")
    mlflow.log_metric("rmse", ridge_rmse)

    mlflow.sklearn.log_model(
        sk_model=ridge_pipeline,
        name=settings.model_artifact_path,
        registered_model_name=settings.registered_model_name,
        serialization_format="skops",
    )


Registered model 'green-taxi-duration' already exists. Creating a new version of this model...
2026/09/24 17:40:21 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: green-taxi-duration, version 4
Created version '4' of model 'green-taxi-duration'.


🏃 View run ridge-regression at: http://127.0.0.1:5001/#/experiments/1/runs/ad1e274f77ca47c3b17beafc31a9a758
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


## batch prdiction with the best Model 

In [26]:
result_path = score_batch_flow(
    input_uri=settings.batch_input_uri,
    output_path=str(output_path),
)

17:45:32.540 | INFO    | Flow run 'shaggy-jackalope' - Beginning flow run 'shaggy-jackalope' for flow 'score-green-taxi-batch'

17:45:32.549 | INFO    | Flow run 'shaggy-jackalope' - View at http://127.0.0.1:4200/runs/flow-run/c87b39be-a41a-43d1-94fa-7d0e4fff58d6

17:45:32.562 | INFO    | Task run 'score_dataframe-045' - Reading input from https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-02.parquet

17:45:33.419 | INFO    | Task run 'score_dataframe-045' - Loading model from models:/green-taxi-duration/4

17:45:34.263 | INFO    | Task run 'score_dataframe-045' - Saved predictions to data/predictions/notebook_batch_predictions.parquet

17:45:34.266 | INFO    | Task run 'score_dataframe-045' - Finished in state Completed()

17:45:34.571 | INFO    | Flow run 'shaggy-jackalope' - Finished in state Completed()

In [27]:
predictions = pd.read_parquet(result_path)

predictions.head()

,ride_id,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance,predicted_duration,model_reference,actual_duration,diff
0,a1bbbf26-4db0-4499-a6f6-f5eac933956c,2025-02-01 00:12:15,166,41,0.65,6.617152,models:/green-taxi-duration/4,3.550000,-3.067152
1,ec418a0b-d9b0-47d8-94ff-ce1b6b89ef1d,2025-01-31 23:57:05,255,161,6.57,24.528947,models:/green-taxi-duration/4,27.316667,2.787720
2,5c50915f-7d22-42ed-8dc8-407c8c52dfdf,2025-02-01 00:24:26,75,182,8.36,23.695008,models:/green-taxi-duration/4,25.466667,1.771659
3,5b321b85-a805-49ce-9350-4fd3978fc9f1,2025-02-01 00:17:15,97,209,2.40,14.588516,models:/green-taxi-duration/4,8.683333,-5.905183
4,2aa88ee1-4266-4030-88d8-6142342a2591,2025-02-01 00:17:36,7,223,1.31,9.220725,models:/green-taxi-duration/4,9.000000,-0.220725


In [28]:
batch_rmse = root_mean_squared_error(

    predictions["actual_duration"],
    predictions["predicted_duration"],

)

print(f"Batch RMSE : {batch_rmse:.2f}")

Batch RMSE : 5.18


### Check FastAPI with Model Version 4 

In [32]:
client = TestClient(app)

client.get("/health").json()

sample_payload = {

    "PULocationID" : 1,
    "DOLocationID" : 2,
    "trip_distance": 3.5,
}

response = client.post("/predict", json=sample_payload)
response.json()

{'prediction': 16.56706163831244, 'model_uri': 'models:/green-taxi-duration/4'}